In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="evaluation",
    notebook_name="02_perplexity.ipynb"
)

# Perplexity: Measuring How "Surprised" a Language Model Is

## What You'll Learn

In this notebook, we'll learn about **perplexity** -- the go-to metric for
evaluating language models. By the end, you'll:

1. Understand what perplexity measures (and why we care)
2. Calculate perplexity **from scratch** using basic Python
3. Understand the math behind it (step by step, no panic!)
4. Use a real language model to compute perplexity on actual text

### The One-Sentence Summary

> **Perplexity = how many words the model is "choosing between" on average.**
> Lower = better. A perplexity of 10 means the model is, on average, confused
> among about 10 possible next words.

---
## 1. Setup

In [ ]:
# Install if needed (uncomment the lines below)
# !pip install torch transformers

import math
import torch
import matplotlib.pyplot as plt

print("Setup complete!")

---
## 2. The Intuition: A Guessing Game

Imagine you're playing a word-guessing game. I say the beginning of a sentence,
and you have to guess the next word.

**Easy sentence:**
- "The cat sat on the ___" → You'd guess "mat" or "floor" (few choices, low surprise)

**Hard sentence:**
- "The quantum ___" → Could be anything! (many choices, high surprise)

Perplexity measures this "surprise" mathematically. Let's build up to it step by step.

---
## 3. Building Perplexity From Scratch

### Step 1: A Toy Language Model

Let's create the simplest possible "language model" -- one that assigns
probabilities to the next word based on a small dictionary.

In [ ]:
# Our toy "language model" -- just a dictionary of word probabilities
# After seeing "the cat", what word comes next?

toy_model = {
    "the": 0.15,      # Very common word, model expected it
    "cat": 0.05,      # Less expected after "the"
    "sat": 0.20,      # After "the cat", "sat" is likely!
    "on": 0.25,       # After "sat", "on" is very likely
    "mat": 0.10,      # After "on the", "mat" is somewhat likely
    "quantum": 0.001, # Very unexpected in this context!
}

print("Our toy model's predictions:")
print("-" * 40)
for word, prob in toy_model.items():
    bar = "█" * int(prob * 100)
    print(f"  '{word:10s}': {prob:.1%} {bar}")

### Step 2: Computing Perplexity Step by Step

The formula for perplexity is:

```
Perplexity = exp( -1/N × Σ ln(P(word_i)) )
```

Don't worry! Let's break it into simple steps:

1. For each word, get the probability the model assigned
2. Take the log of each probability
3. Average the logs
4. Negate and exponentiate

In [ ]:
# Let's compute perplexity for a "normal" sentence
sentence_normal = ["the", "cat", "sat", "on"]

print("=" * 60)
print("Computing perplexity for: 'the cat sat on'")
print("=" * 60)

# Step 1: Get probabilities
print("\nStep 1: Get model's probability for each word")
probs = [toy_model[word] for word in sentence_normal]
for word, prob in zip(sentence_normal, probs):
    print(f"  P('{word}') = {prob}")

# Step 2: Take the log of each
print("\nStep 2: Take the natural log (ln) of each probability")
log_probs = [math.log(p) for p in probs]
for word, lp in zip(sentence_normal, log_probs):
    print(f"  ln(P('{word}')) = {lp:.4f}")

# Step 3: Average them
avg_log_prob = sum(log_probs) / len(log_probs)
print(f"\nStep 3: Average the log probabilities")
print(f"  Average = {sum(log_probs):.4f} / {len(log_probs)} = {avg_log_prob:.4f}")

# Step 4: Negate and exponentiate
perplexity = math.exp(-avg_log_prob)
print(f"\nStep 4: Negate and exponentiate")
print(f"  Perplexity = exp(-({avg_log_prob:.4f})) = exp({-avg_log_prob:.4f}) = {perplexity:.2f}")

print(f"\n>>> RESULT: Perplexity = {perplexity:.2f}")
print(f"    This means the model was choosing between ~{perplexity:.0f} words on average.")

In [ ]:
# Now let's try a "weird" sentence with an unexpected word
sentence_weird = ["the", "cat", "sat", "quantum"]

print("=" * 60)
print("Computing perplexity for: 'the cat sat quantum'")
print("=" * 60)

probs_weird = [toy_model[word] for word in sentence_weird]
log_probs_weird = [math.log(p) for p in probs_weird]
avg_log_prob_weird = sum(log_probs_weird) / len(log_probs_weird)
perplexity_weird = math.exp(-avg_log_prob_weird)

for word, prob in zip(sentence_weird, probs_weird):
    surprise = "(expected)" if prob > 0.05 else "(VERY SURPRISING!)"
    print(f"  P('{word}') = {prob} {surprise}")

print(f"\n>>> RESULT: Perplexity = {perplexity_weird:.2f}")
print(f"    The model was choosing between ~{perplexity_weird:.0f} words on average.")
print(f"\n    Compare: normal sentence = {perplexity:.2f}, weird sentence = {perplexity_weird:.2f}")
print(f"    The weird sentence has MUCH higher perplexity (more surprise)!")

---
## 4. Let's Make a Reusable Function

Now that we understand the steps, let's wrap it into a clean function:

In [ ]:
def compute_perplexity(probabilities):
    """
    Compute perplexity from a list of probabilities.

    Args:
        probabilities: list of floats, each between 0 and 1.
                       These are the model's predicted probability for each word.

    Returns:
        float: the perplexity score (lower = better)
    """
    n = len(probabilities)
    log_prob_sum = sum(math.log(p) for p in probabilities)
    avg_neg_log_prob = -log_prob_sum / n
    return math.exp(avg_neg_log_prob)


# Test it!
print("Testing our function:")
print(f"  'the cat sat on':      Perplexity = {compute_perplexity([0.15, 0.05, 0.20, 0.25]):.2f}")
print(f"  'the cat sat quantum': Perplexity = {compute_perplexity([0.15, 0.05, 0.20, 0.001]):.2f}")

---
## 5. Visualizing How Probability Affects Perplexity

In [ ]:
# What happens to perplexity as model confidence changes?
# Let's vary the probability of a single word and see

import numpy as np

probs_range = np.linspace(0.01, 0.99, 100)
perplexities = [compute_perplexity([p]) for p in probs_range]

plt.figure(figsize=(10, 5))
plt.plot(probs_range, perplexities, linewidth=2, color="#2196F3")
plt.xlabel("Model's Probability for the Word", fontsize=12)
plt.ylabel("Perplexity", fontsize=12)
plt.title("How Probability Affects Perplexity\n(Single word example)", fontsize=14, fontweight="bold")

# Add annotations
plt.annotate("Model is guessing\n(low confidence)",
             xy=(0.1, compute_perplexity([0.1])),
             xytext=(0.25, 60),
             arrowprops=dict(arrowstyle="->"), fontsize=10)
plt.annotate("Model is confident\n(high confidence)",
             xy=(0.9, compute_perplexity([0.9])),
             xytext=(0.6, 30),
             arrowprops=dict(arrowstyle="->"), fontsize=10)

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Key insight: As the model becomes more confident (higher probability),")
print("perplexity drops dramatically. A confident model has low perplexity.")

---
## 6. Comparing Sentences Side by Side

In [ ]:
# Let's compare the perplexity of different sentence types

sentences = {
    "Common phrase\n'the cat sat on'": [0.15, 0.10, 0.20, 0.30],
    "Somewhat unusual\n'the old cat sat'": [0.15, 0.05, 0.10, 0.20],
    "Rare combination\n'the quantum cat danced'": [0.15, 0.001, 0.05, 0.01],
    "Near random\n'purple sings the umbrella'": [0.01, 0.005, 0.15, 0.005],
}

names = list(sentences.keys())
perplexities = [compute_perplexity(probs) for probs in sentences.values()]
colors = ["#4CAF50", "#FFC107", "#FF9800", "#F44336"]

plt.figure(figsize=(10, 5))
bars = plt.bar(names, perplexities, color=colors, edgecolor="black", linewidth=0.5)
plt.ylabel("Perplexity (lower = better)", fontsize=12)
plt.title("Perplexity of Different Sentences\n(Our toy model)", fontsize=14, fontweight="bold")

# Add value labels on bars
for bar, ppl in zip(bars, perplexities):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
             f"{ppl:.1f}", ha="center", fontweight="bold", fontsize=11)

plt.tight_layout()
plt.show()

print("Notice how perplexity increases as sentences get weirder!")
print("The model is more 'surprised' by unusual word combinations.")

---
## 7. Real Language Model: Computing Perplexity with GPT-2

Now let's use a REAL language model (GPT-2, a smaller predecessor to ChatGPT)
to compute perplexity on actual sentences.

> **Note:** This section requires `transformers` and `torch`. If you don't
> have them installed, the earlier sections still work fine on their own.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load GPT-2 (small version, ~500MB download on first run)
print("Loading GPT-2 model (this may take a moment on first run)...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()  # Set to evaluation mode
print("Model loaded!")

In [ ]:
def gpt2_perplexity(text):
    """
    Compute the perplexity of a text string using GPT-2.

    How it works:
    1. Convert text to tokens (the model reads tokens, not words)
    2. Feed tokens to GPT-2 and get predicted probabilities
    3. Compute cross-entropy loss (average negative log probability)
    4. Perplexity = exp(loss)
    """
    # Tokenize the text
    encodings = tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids

    # Get model's predictions
    with torch.no_grad():  # No need to compute gradients (we're not training)
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss  # Cross-entropy loss = average negative log prob

    # Perplexity = exp(loss)
    ppl = math.exp(loss.item())
    return ppl


print("Function ready! Let's test it.")

In [ ]:
# Test with different sentences
test_sentences = [
    "The cat sat on the mat.",
    "The president of the United States gave a speech today.",
    "She walked to the store and bought some groceries.",
    "The quantum fluctuation of the interdimensional portal surprised the aardvark.",
    "Colorless green ideas sleep furiously.",  # Famous linguistically correct but meaningless sentence
    "asdf qwerty zxcv bloop fnord.",  # Nonsense
]

print("GPT-2 Perplexity for Different Sentences")
print("=" * 70)

results = []
for sentence in test_sentences:
    ppl = gpt2_perplexity(sentence)
    results.append((sentence, ppl))
    print(f"\n  Perplexity: {ppl:>10.2f}  |  \"{sentence}\"")

print("\n" + "=" * 70)
print("\nNotice: Normal sentences have LOW perplexity (model expected them).")
print("Weird/nonsense sentences have HIGH perplexity (model is surprised).")

In [ ]:
# Visualize the results
short_labels = [
    "Cat on mat",
    "President speech",
    "Store groceries",
    "Quantum aardvark",
    "Colorless green\nideas",
    "Pure nonsense",
]
ppls = [r[1] for r in results]

# Color based on perplexity level
colors = []
for p in ppls:
    if p < 50:
        colors.append("#4CAF50")  # green = good
    elif p < 200:
        colors.append("#FFC107")  # yellow = okay
    elif p < 1000:
        colors.append("#FF9800")  # orange = high
    else:
        colors.append("#F44336")  # red = very high

plt.figure(figsize=(12, 5))
bars = plt.bar(short_labels, ppls, color=colors, edgecolor="black", linewidth=0.5)
plt.ylabel("Perplexity (lower = model is less surprised)", fontsize=11)
plt.title("GPT-2 Perplexity on Different Sentences", fontsize=14, fontweight="bold")
plt.yscale("log")  # Log scale because values vary hugely

# Add value labels
for bar, ppl in zip(bars, ppls):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.1,
             f"{ppl:.0f}", ha="center", fontweight="bold", fontsize=10)

plt.tight_layout()
plt.show()

---
## 8. Experiment: Try Your Own Sentences!

Change the sentences below and see how GPT-2's perplexity changes.

**Ideas to try:**
- A simple, everyday sentence
- A sentence in another language
- Song lyrics
- Random keyboard mashing
- A very long sentence vs a short one

---
## What You Just Built

You now understand perplexity — the go-to metric for language models:
1. **From scratch:** log probs → average → exponentiate = perplexity
2. **Intuition confirmed:** normal sentences → low perplexity, weird sentences → high perplexity
3. **Real model verification:** GPT-2's perplexity matches our expectations

For the full mathematical derivation (cross-entropy connection, vocabulary dependence, BPE effects) and staff-level interview questions, see the [interview deep-dive](./perplexity-interview.md).

Next up: [BLEU & ROUGE notebook](./03_bleu_rouge.ipynb) for text generation metrics.

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)

---
## 9. Key Takeaways

```
┌─────────────────────────────────────────────────────┐
│              Perplexity Cheat Sheet                  │
│                                                     │
│  What:     How "surprised" a model is by text       │
│  Formula:  exp(average negative log probability)    │
│  Lower:    Better (model predicts text well)        │
│  Higher:   Worse (model is confused)                │
│                                                     │
│  Think of it as: "How many words is the model       │
│  choosing between at each step?"                    │
│                                                     │
│  Perplexity of 1   = Perfect prediction             │
│  Perplexity of 10  = Choosing among ~10 words       │
│  Perplexity of 100 = Choosing among ~100 words      │
│                                                     │
│  Caution: Only compare on the SAME dataset!         │
└─────────────────────────────────────────────────────┘
```

### What We Covered

1. Perplexity measures how well a model predicts text
2. We computed it from scratch: log probs → average → exponentiate
3. Normal sentences → low perplexity, weird sentences → high perplexity
4. Real GPT-2 model confirms: it's less surprised by common English

### Next Steps

- Read the [Perplexity guide](../metrics/perplexity.md) for more details
- Try the [BLEU & ROUGE notebook](./03_bleu_rouge.ipynb) for text generation metrics